In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Define the quantization config (4-bit in this case)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_quant_type="nf4" # Normal Float 4 (optimized for weights)
)

xlstm_config = AutoConfig.from_pretrained("NX-AI/xLSTM-7b")
xlstm_config.step_kernel = "native"
xlstm_config.chunkwise_kernel = "chunkwise--native_autograd"
xlstm_config.sequence_kernel = "native_sequence__native"

xlstm = AutoModelForCausalLM.from_pretrained("NX-AI/xLSTM-7b",
                                             config=xlstm_config, device_map="auto", quantization_config=quantization_config)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("NX-AI/xLSTM-7b")

# Your prompt
prompt = "Explain quantum computing in simple terms."

# Tokenize and send to the same device as the model
inputs = tokenizer(prompt, return_tensors="pt")['input_ids'].to(xlstm.device)

# Get the BOS token ID from the tokenizer
bos_id = tokenizer.bos_token_id

# Prepend BOS
bos_tensor = torch.tensor([[bos_id]], device=xlstm.device, dtype=inputs.dtype)
tokens_with_bos = torch.cat([bos_tensor, inputs], dim=1)

# Generate
outputs = xlstm.generate(
    tokens_with_bos,
    max_new_tokens=200,   # adjust for output length
    temperature=0.7,      # randomness
    top_p=0.9,            # nucleus sampling
    do_sample=True
)

# Decode and print
print(tokenizer.decode(outputs[0]))


Loading weights: 100%|██████████| 483/483 [01:20<00:00,  5.97it/s]
d:\A_Facultate\Diseration\logit_lens\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


<|endoftext|>Explain quantum computing in simple terms.
Quantum computing is a type of computing that uses quantum mechanics to perform complex calculations at a faster rate. Quantum computers use quantum bits (qubits) instead of the traditional binary bits used in classical computing. Qubits can exist in multiple states at the same time, allowing quantum computers to perform multiple calculations at the same time. This makes quantum computers more powerful than classical computers for certain types of problems.<|endoftext|>#!/usr/bin/python3
#
# This example is public domain.
#

import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

def get_value_of_cos(angle):
    return np.cos(angle)

def get_value_of_sin(angle):
    return np.sin(angle)

def get_value_of_tan(angle):
    return np.tan(angle)

def


In [10]:
import lm_eval
from lm_eval.models.huggingface import HFLM

# 1. Your existing setup code (xlstm and tokenizer) goes here...
# [Assuming 'xlstm' and 'tokenizer' are already defined as in your snippet]

# 2. Wrap your specific instance for the harness
# We pass the pre-initialized model and tokenizer directly
lm_eval_model = HFLM(
    pretrained=xlstm,        # Pass the object, not a string path
    tokenizer=tokenizer, 
    limit=10,  
    batch_size="auto"           # Recurrent models often prefer smaller batches
)

# 3. Run the benchmark programmatically
results = lm_eval.simple_evaluate(
    model=lm_eval_model,
    tasks=["arc_challenge", "winogrande"], 
    num_fewshot=0,    
    device="cuda"
)



`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
Overwriting default num_fewshot of winogrande from None to 0
Overwriting default num_fewshot of arc_challenge from None to 0

100%|██████████| 1267/1267 [00:00<00:00, 80922.54it/s]














100%|██████████| 1172/1172 [00:01<00:00, 849.19it/s]



Passed argument batch_size = auto:1. Detecting largest batch size


Running loglikelihood requests:   0%|          | 0/40168 [19:30<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
import json
from lm_eval.utils import make_table
# 1. Generate and print the readable console table
table_str = make_table(results)
print(table_str)

# 2. Save the table as a text file for easy reading
with open("hellaswag_results_table.txt", "w", encoding="utf-8") as f:
    f.write(table_str)
# 3. Save the full, raw results as a JSON file (useful if you need exact metric numbers later)
with open("hellaswag_raw_results.json", "w", encoding="utf-8") as f:
    # default=str ensures we don't get errors if there are unusual number formats
    json.dump(results, f, indent=4, default=str)